In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.DataFrame({
    'Client': ['Mrs. Sarah Connor', '  JOHN   WICK  ', None, 'mr. walter white', 'JESSE PINKMAN'],
    'Income': ['$120,000', '€85.500', '90 000 руб.', None, '45000'],
    'Discount': ['10%', '5 percent', '15 %', '0', None],
    'Responded': ['yes', 'NO', None, '1', '0']
})

print(df)

              Client       Income   Discount Responded
0  Mrs. Sarah Connor     $120,000        10%       yes
1      JOHN   WICK        €85.500  5 percent        NO
2                NaN  90 000 руб.       15 %       NaN
3   mr. walter white          NaN          0         1
4      JESSE PINKMAN        45000        NaN         0


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Client     4 non-null      str  
 1   Income     4 non-null      str  
 2   Discount   4 non-null      str  
 3   Responded  4 non-null      str  
dtypes: str(4)
memory usage: 292.0 bytes


In [ ]:
df.head()

,Client,Income,Discount,Responded
0,Mrs. Sarah Connor,"$120,000",10%,yes
1,JOHN WICK,€85.500,5 percent,NO
2,NaN,90 000 руб.,15 %,NaN
3,mr. walter white,NaN,0,1
4,JESSE PINKMAN,45000,NaN,0


In [ ]:



def clean_income(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r'[^0-9]', '', x)
    
    return float(x)

df['income_clean'] = df['Income'].apply(clean_income)
df['income_clean']

0    12000000.0
1     8550000.0
2     9000000.0
3           NaN
4     4500000.0
Name: income_clean, dtype: float64

Очищаем числа от мусорных значений

In [ ]:
def clean_discount(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r'[^0-9]', '', x)

    return float(x)

df['discount_clean'] = df['Discount'].apply(clean_discount)
df['discount_clean']

0    10.0
1     5.0
2    15.0
3     0.0
4     NaN
Name: discount_clean, dtype: float64

Очищаем числа от мусорных значений

In [ ]:
# %%

def clean_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    x = re.sub(r'\b(mr|mrs|ms)\b\.?\s*', '', x, flags=re.IGNORECASE)
    x = re.sub(r'\s+', ' ', x)
    x = x.strip()
    x = x.title()
    return x

df['name_clean'] = df['Client'].apply(clean_name)
df['name_clean']

    

0     Sarah Connor
1        John Wick
2              NaN
3     Walter White
4    Jesse Pinkman
Name: name_clean, dtype: str

Приводим имена к одному стандарту

In [ ]:

def clean_responded(x):
    if pd.isna(x):
        return np.nan
    x = str(x).lower()
    if x in ('yes', '1'):
        return True
    elif x in ('no', '0'):
        return False
    return np.nan

df['responded_clean'] = df['Responded'].apply(clean_responded)
df['responded_clean']

0     True
1    False
2      NaN
3     True
4    False
Name: responded_clean, dtype: object

Приводим значения к Правда или Ложь

In [ ]:

def clean_dataframe(df, column_map):
    for col, func in column_map.items():
        if col in df.columns:
            df[col] = df[col].apply(func)
        else:
            print(f'Такой колонки "{col}" нет - пропускаю')
        return df

mapping = {
    'Client': clean_name,
    'Income': clean_income,
    'Discount': clean_discount,
    'Responded': clean_responded
}

df

df_clean = clean_dataframe(df, mapping)
df_clean 

,Client,Income,Discount,Responded,responded_clean
0,Sarah Connor,"$120,000",10%,yes,True
1,John Wick,€85.500,5 percent,NO,False
2,NaN,90 000 руб.,15 %,NaN,NaN
3,Walter White,NaN,0,1,True
4,Jesse Pinkman,45000,NaN,0,False


Создаем универсальную функцию